### 📖 Hướng dẫn sử dụng công cụ Giải nén / Liệt kê File

Dưới đây là giải thích cho các thông số cấu hình:

*   **🛠️ OPERATION MODE (Chế độ hoạt động):**
    *   `LIST_FILES`: Chỉ xem danh sách các file và cấu trúc thư mục bên trong file nén (không tốn dung lượng ổ đĩa).
    *   `EXTRACT`: Thực hiện giải nén file vào Google Drive.
*   **🔗 INPUT FILE (Đường dẫn File):**
    *   Dán link Google Drive (chia sẻ ở chế độ 'Bất kỳ ai có liên kết') hoặc ID của file nén bạn muốn xử lý.
    Ví dụ: https:\/\/drive.google.com/file/d/**1LKDCZsukhqyB9bV7jSfUZj2M39gZg_Z3**/view hoặc **1LKDCZsukhqyB9bV7jSfUZj2M39gZg_Z3** đều được.
*   **⚙️ EXTRACTION SETTINGS (Cài đặt giải nén - Chỉ áp dụng cho chế độ EXTRACT):**
    *   `CREATE_OUTPUT_FOLDER`:
        *   `YES`: Tự động tạo một thư mục mới để chứa các file giải nén ra (tránh làm lộn xộn thư mục gốc).
        *   `NO`: Giải nén trực tiếp vào thư mục chứa file nén.
    *   `OUTPUT_FOLDER_NAME`: Tên thư mục bạn muốn tạo. Nếu để trống, hệ thống sẽ lấy tên file nén làm tên thư mục.
*   **📂 STORAGE SETTINGS:**
    *   `STORAGE_TYPE`: Chọn nơi lưu trữ là Drive cá nhân (`MyDrive`) hoặc Bộ nhớ dùng chung (`Shared Drives`).

---

In [1]:
import os, re, warnings
from google.colab import drive, auth
from googleapiclient.discovery import build

warnings.filterwarnings("ignore")

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
auth.authenticate_user()

# --- FORM INPUTS ---
# @markdown ### 🛠️ OPERATION MODE
MODE = "LIST_FILES" # @param ["LIST_FILES", "EXTRACT"]
# @markdown ### 🔗 INPUT FILE (ZIP, GZ, TAR or TAR.GZ)
FILE_URL_OR_ID = "https://drive.google.com/file/d/1LKDCZsukhqyB9bV7jSfUZj2M39gZg_Z3/view?usp=drive_link" # @param {type:"string"}
# @markdown ### ⚙️ EXTRACTION SETTINGS (Only for EXTRACT mode)
CREATE_OUTPUT_FOLDER = "YES" # @param ["YES", "NO"]
OUTPUT_FOLDER_NAME = "" # @param {type:"string"}
# @markdown ### 📂 STORAGE SETTINGS
STORAGE_TYPE = "MyDrive" # @param ["MyDrive", "Shared Drives"]

def extract_id(input_value):
    id_pattern = r'(?<=/d/)([a-zA-Z0-9_-]+)'
    match = re.search(id_pattern, input_value)
    if match: return match.group(1)
    return input_value.strip()

def get_real_path(service, folder_id):
    curr_id = folder_id
    path_segments = []
    while True:
        try:
            f = service.files().get(fileId=curr_id, fields='name, parents', supportsAllDrives=True).execute()
            name = f.get('name')
            if name in ['My Drive', 'Drive của tôi']:
                path_segments.insert(0, '/content/drive/MyDrive')
                break
            path_segments.insert(0, name)
            if not f.get('parents'): break
            curr_id = f.get('parents')[0]
        except:
            path_segments.insert(0, '/content/drive/MyDrive')
            break
    return "/".join(path_segments).replace("//", "/")

def process_file():
    if not FILE_URL_OR_ID:
        print("❌ ERROR: Please enter a File ID or URL.")
        return

    service = build('drive', 'v3')
    file_id = extract_id(FILE_URL_OR_ID)

    try:
        meta = service.files().get(fileId=file_id, fields='name, parents', supportsAllDrives=True).execute()
        fname = meta.get('name').strip()
        parent_id = meta.get('parents')[0]
        parent_path = get_real_path(service, parent_id)
        full_input_path = os.path.join(parent_path, fname)

        if not os.path.exists(full_input_path):
            print(f"❌ ERROR: File not found at {full_input_path}")
            return

        if MODE == "LIST_FILES":
            print(f"📋 Listing contents of: {fname}\n" + "-"*40)
            if fname.lower().endswith('.zip'):
                !unzip -l "{full_input_path}"
            elif fname.lower().endswith(('.tar.gz', '.tgz', '.tar')):
                !tar -tf "{full_input_path}"
            else:
                print("Attempting list with 7z...")
                !7z l "{full_input_path}"

        else: # EXTRACT MODE
            print(f"🆔 File identified: {fname}")
            if CREATE_OUTPUT_FOLDER == "YES":
                base_name = fname
                for ext in ['.tar.gz', '.gz', '.zip', '.tgz', '.tar']:
                    if base_name.lower().endswith(ext):
                        base_name = base_name[: -len(ext)]
                        break
                final_folder = OUTPUT_FOLDER_NAME if OUTPUT_FOLDER_NAME.strip() else base_name
                target_path = os.path.join(parent_path, final_folder)
                if not os.path.exists(target_path): os.makedirs(target_path)
            else:
                target_path = parent_path

            print(f"⚡ Processing extraction to: {target_path}")
            if fname.lower().endswith('.zip'):
                !unzip -o -q "{full_input_path}" -d "{target_path}"
            elif fname.lower().endswith(('.tar.gz', '.tgz')):
                !tar -xzf "{full_input_path}" -C "{target_path}"
            elif fname.lower().endswith('.tar'):
                !7z x "{full_input_path}" -o"{target_path}" -y
            elif fname.lower().endswith('.gz'):
                dest_file = os.path.join(target_path, fname)
                !cp "{full_input_path}" "{dest_file}"
                !gunzip -f "{dest_file}"
            else:
                !7z x "{full_input_path}" -o"{target_path}" -y

            print("-" * 40)
            print(f"🎉 SUCCESS! Location: {target_path}")

    except Exception as e:
        print(f"❌ SYSTEM ERROR: {e}")

process_file()

Mounted at /content/drive


📋 Listing contents of: FileZipToDownload.zip
----------------------------------------
Archive:  /content/drive/MyDrive/Temp/TestColabUnzip/FileZipToDownload.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-04-29 13:48   FileZipToDownload/
    13682  2026-03-29 23:21   FileZipToDownload/DocFile.docx
      538  2026-03-29 23:17   FileZipToDownload/DownloadFileList.csv
     8559  2026-03-29 23:21   FileZipToDownload/ExcelFile.xlsx
       81  2026-03-29 23:20   FileZipToDownload/TextFile.txt
      231  2026-03-29 23:20   FileZipToDownload/ZipFile.zip
---------                     -------
    23091                     6 files
